# Practice # 6
plan > agent > excute > report application



## create_react_agent 과 create_tool_calling_agent 의 차이
create_react_agent와 create_tool_calling_agent는 모두 LangGraph/LangChain에서 LLM 기반 에이전트를 생성하는 함수지만, 에이전트의 reasoning 과정 관리 방식과 투명성에서 큰 차이가 있습니다

### create_react_agent : 
- 외부로 Thought/Action/Observation 순서 노출
- *course correction(경로 수정) 자유로움 : 루프 과정과 중간 reasoning 흐름 확인·개입 가능

### create_tool_calling_agent :
- reasoning은 내부적으로만 진행됨
- 경로 수정 불가 : 자동화·빠른 워크플로우에 적합 : 과정 숨김; 결과만 확인 가능, 디버깅 불편



## 구성도
![구성도]()



In [1]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from typing import List

In [2]:
strt_langsmith('practive_6')

LangSmith 추적을 시작합니다.
[프로젝트명]
practive_6


In [3]:
from typing import TypedDict, Annotated, List, Literal,Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage
from langgraph.prebuilt import create_react_agent

### Agent

In [4]:
class State(TypedDict):
    question : Annotated[str,'user input question']   # 사용자 질의 or requeustion 질의
    plan : Annotated[list[str],'get plan_node']  # llm 생성한 작업 계획서
    messages : Annotated[list,add_messages]      # 작업 수행 후 얻은 데이터
    past_steps :Annotated[list, add_messages]
    answer : Annotated[str,' output final answer'] # 최종 답변 출력

In [5]:
class Plan(BaseModel):
    """Sorted steps to execute the plan"""
    steps: Annotated[List[str],"Different steps to follow, should be in sorted order"]

class Response(BaseModel):
    """Response to user."""
    # 사용자 응답
    response: str

class Act(BaseModel):
    """Action to perform."""
    # LLM 이 판단하여 Reponse 인지 Plan 인지 선택하는 모델
    # 만약 Response 인경우 response 변수에 답변 생성
    # 그외 Plan 인경우 step 변수에 답변 생성
    action: Union[Response, Plan] = Field(
        description="Action to perform. If you want to respond to user, use Response. "
        "If you need to further use tools to get the answer, use Plan."
    )

In [6]:
def get_agent():
    web_search = get_tavily_tool()
    prompt = get_prompt_agent()
    return create_react_agent(get_gemini(),[web_search],prompt=prompt) 

### node define

In [7]:
## 계획 수립
def plan_node(state : State)-> State:
    question = state['question']
    prompt = get_prompt_planner()
    llm = get_gemini()
    llm_with_plan = llm.with_structured_output(Plan)
    chain = prompt | llm_with_plan
    plan = chain.invoke({'messages':question})
    return State({'plan':plan.steps})

## 계획 [0] 꺼내서 툴 사용
def execute_agent_node(state : State)->State : 
    agent = get_agent()
    plan = state['plan']
    plan_str = "\n".join(f"{i+1}. {text}" for i, text in enumerate(plan))
    task = plan[0]
    task_str = f"""For the following plan: \n\n {plan_str}\n\n You are tasked with executing [step 1. {task}]."""
    agent_response = agent.invoke({'messages':[task_str]})
    return State({'past_steps':[f" Question :{task}\n Response :{agent_response['messages'][-1].content}"]})

## 마지막 작업, 요구사항, 계획 꺼내서 Plan 인지 Response 인지 판단
def replan_node(state:State)-> State:
    # 판단 llm
    prompt = get_prompt_replanner()
    chain = prompt | get_gpt().with_structured_output(Act)
    outputs =  chain.invoke({'input':state['question'],'plan':state['plan'],'past_steps':state['past_steps']})
    print(outputs)
    # outputs => action=Plan(steps=['RAG의 작동 방식을 설명합니다.', 'RAG의 장단점을 설명합니다.', 'RAG의 활용 사례를 설명합니다.'])
    if isinstance(outputs.action,Response):
        return State({'answer':outputs.action.response})
    else:  # result == Plan 
        next_plan = outputs.action.steps
        if len(next_plan) == 0:
            return {"answer": "No more steps needed."}
        else:
            return {"plan": next_plan}



def final_generate_node(state: State):
    final_report = get_prompt_generate_markdown() | get_gemini() | StrOutputParser()
    answer = final_report.invoke({"input": state["question"], "past_steps": state['past_steps']})
    return {"answer": answer}

### node condition

In [8]:
# action=Plan(steps=['각 핵심 요소인 검색, 생성, 통합에 대해 자세한 내용을 설명한다.', 'RAG 개발의 전반적인 중요성을 강조하며 마무리한다.'])
def is_replan(state:State)->Literal['final_generate_node','execute_agent_node']:
    if "answer" in state and state["answer"]:
        return 'final_generate_node'
    else:
        return 'execute_agent_node'

### graph define

In [9]:
state_graph = StateGraph(State)
state_graph.add_node('plan_node',plan_node)
state_graph.add_node('execute_agent_node',execute_agent_node)
state_graph.add_node('replan_node',replan_node)
state_graph.add_node('final_generate_node',final_generate_node)

state_graph.add_edge(START,'plan_node')
state_graph.add_edge('plan_node','execute_agent_node')
state_graph.add_edge('execute_agent_node','replan_node')

# state_graph.add_edge('replan_node',END)
state_graph.add_conditional_edges(
    source='replan_node',
    path=is_replan
)
state_graph.add_edge('final_generate_node',END)
ck = get_check_pointer()
graph = state_graph.compile(checkpointer=ck)

In [10]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan_node(plan_node)
	execute_agent_node(execute_agent_node)
	replan_node(replan_node)
	final_generate_node(final_generate_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan_node;
	execute_agent_node --> replan_node;
	plan_node --> execute_agent_node;
	replan_node -.-> execute_agent_node;
	replan_node -.-> final_generate_node;
	final_generate_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### 부분 실행

In [11]:
# plan = plan_node({'question':['RAG란']})
# act = execute_agent_node({'plan':plan['plan']})
# inputs = State({'input':['RAG란'], 'plan':plan['plan'],'past_steps':act})
# replan = replan_node(inputs)
# print(replan)


### 최종

In [29]:
uuid = get_random_uuid()
config =get_runnable_config(recursion_limit=20,thread_id=uuid)

In [30]:

question = 'RAG 란?'
invoke_graph(graph,{"question": [question]},config)


🔄 Node: plan_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
RAG에 대한 정의를 설명합니다.
RAG의 작동 방식을 설명합니다.
RAG의 장단점을 설명합니다.
RAG의 활용 사례를 설명합니다.

🔄 Node: agent in [execute_agent_node] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

RAG(Retrieval-Augmented Generation)은 검색 증강 생성으로, 대규모 언어 모델(LLM)이 외부 지식 베이스에서 관련 정보를 검색하여 답변을 생성하는 기술입니다.

🔄 Node: execute_agent_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
 Question :RAG에 대한 정의를 설명합니다.
 Response :RAG(Retrieval-Augmented Generation)은 검색 증강 생성으로, 대규모 언어 모델(LLM)이 외부 지식 베이스에서 관련 정보를 검색하여 답변을 생성하는 기술입니다.
action=Plan(steps=['RAG의 작동 방식을 설명합니다.', 'RAG의 장단점을 설명합니다.', 'RAG의 활용 사례를 설명합니다.'])

🔄 Node: replan_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
RAG의 작동 방식을 설명합니다.
RAG의 장단점을 설명합니다.
RAG의 활용 사례를 설명합니다.

🔄 Node: agent in [execute_agent_node] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Messa

In [14]:
snapshot = graph.get_state(config)
snapshot.values

{'question': ['2025년 10월 4일 서울 근교 데이트 코스 및 드라이브 코스 작성해줘( 1시쯤 만나서 점심먹을거야 ) '],
 'plan': ['2025년 10월 4일은 토요일이므로, 서울 근교에서 즐길 수 있는 데이트 및 드라이브 코스를 계획합니다.',
  '오후 1시 서울 근교에서 만나 점심 식사를 할 수 있는 맛집을 추천합니다.',
  '점심 식사 후 가볍게 산책하거나 둘러볼 수 있는 서울 근교의 공원 또는 관광지를 추천합니다.',
  '드라이브 코스로 적합한 서울 근교의 아름다운 길이나 경치 좋은 곳을 추천합니다.',
  '저녁 식사 장소 또는 카페를 추천하며 데이트 코스를 마무리합니다.'],
 'messages': [],
 'past_steps': [HumanMessage(content=' Question :2025년 10월 4일은 토요일이므로, 서울 근교에서 즐길 수 있는 데이트 및 드라이브 코스를 계획합니다.\n Response :2025년 10월 4일 토요일에 맞춰 서울 근교 데이트 및 드라이브 코스를 계획해 드릴게요.\n\n먼저, 10월 초는 가을이 무르익어 날씨가 선선하고 단풍도 아름다운 시기이니, 이를 고려하여 코스를 구성하는 것이 좋겠습니다.\n\n**데이트 및 드라이브 코스 (예시)**\n\n1.  **오후 1시: 점심 식사 (양평 또는 남양주)**\n    *   서울 근교에서 맛집으로 유명한 양평이나 남양주 지역을 추천합니다. 이 지역들은 드라이브 코스로도 인기가 많으며, 맛있는 음식점들이 많습니다.\n    *   **추천 메뉴:** 한정식, 브런치 카페, 분위기 좋은 파스타/스테이크 전문점 등\n\n2.  **점심 식사 후: 산책 및 관광 (두물머리 또는 북한강변)**\n    *   **두물머리:** 양평에 위치한 두물머리는 아름다운 강변 풍경과 느티나무로 유명한 곳입니다. 가볍게 산책하며 사진 찍기 좋습니다.\n    *   **북한강변 산책로:** 남양주나 양평 지역의 북한강변을 따라 조성된 산책로를 걸으며 여

# 검색 증강 생성(RAG) 기술 보고서

## 1. 서론

본 보고서는 검색 증강 생성(Retrieval-Augmented Generation, RAG) 기술에 대한 전반적인 이해를 돕기 위해 작성되었습니다. RAG는 대규모 언어 모델(LLM)이 외부 지식 베이스에서 관련 정보를 검색하여 답변을 생성하는 혁신적인 기술로, 최근 인공지능 분야에서 주목받고 있습니다. 본 보고서에서는 RAG의 정의, 작동 방식, 장단점 및 주요 활용 사례를 상세히 다룹니다.

## 2. RAG의 정의

RAG(Retrieval-Augmented Generation)는 **검색 증강 생성**으로 번역되며, 대규모 언어 모델(LLM)이 학습 데이터에 포함되지 않은 최신 정보나 특정 도메인의 전문 지식을 활용하여 보다 정확하고 풍부한 답변을 생성할 수 있도록 하는 기술입니다. 이는 LLM이 외부 지식 베이스에서 관련 정보를 능동적으로 검색하고, 이를 바탕으로 답변을 생성하는 방식으로 작동합니다.

## 3. RAG의 작동 방식

RAG 시스템은 다음과 같은 세 가지 주요 단계로 작동합니다.

1.  **검색(Retrieval)**: 사용자의 질문이 입력되면, RAG 시스템은 먼저 질문과 관련된 정보를 외부 지식 베이스(예: 문서, 데이터베이스)에서 검색합니다. 이 과정에서 임베딩 기술 등을 활용하여 질문과 가장 유사한 정보를 효율적으로 찾아냅니다.
2.  **증강(Augmentation)**: 검색된 정보는 원래의 질문과 함께 LLM의 입력으로 제공됩니다. 즉, LLM은 단순히 질문에 대한 답변을 생성하는 것을 넘어, 검색된 관련 정보를 참고 자료로 활용하게 됩니다.
3.  **생성(Generation)**: LLM은 증강된 정보, 즉 질문과 검색된 관련 정보를 종합적으로 고려하여 최종 답변을 생성합니다. 이 과정을 통해 LLM은 학습 데이터의 한계를 극복하고 더욱 정확하고 신뢰할 수 있는 답변을 제공할 수 있습니다.

## 4. RAG의 장단점

### 4.1. 장점

*   **최신 정보 반영**: 외부 지식 베이스를 실시간으로 검색하므로 학습 데이터에 포함되지 않은 최신 정보를 답변에 반영할 수 있습니다.
*   **환각(Hallucination) 감소**: 검색된 사실에 기반하여 답변을 생성하므로, LLM이 사실이 아닌 정보를 생성하는 환각 현상을 줄일 수 있습니다.
*   **출처 명확성**: 답변의 근거가 되는 검색된 정보를 제시할 수 있어 답변의 신뢰성을 높입니다.
*   **특정 도메인 지식 강화**: 특정 도메인의 전문 지식을 포함하는 지식 베이스를 활용하여 해당 분야에 대한 LLM의 답변 정확도를 향상시킬 수 있습니다.
*   **비용 효율성**: LLM을 처음부터 재학습시키는 것보다 외부 지식 베이스 구축 및 검색 기능 추가가 더 비용 효율적일 수 있습니다.

### 4.2. 단점

*   **검색 품질 의존성**: 검색 시스템의 성능이 낮을 경우, 관련 없는 정보가 검색되어 답변의 품질이 저하될 수 있습니다.
*   **검색 및 생성 시간 소요**: 정보를 검색하고 이를 바탕으로 답변을 생성하는 과정에서 추가적인 시간이 소요될 수 있습니다.
*   **복잡성 증가**: 기존 LLM 모델에 검색 기능을 통합해야 하므로 시스템의 복잡성이 증가합니다.
*   **지식 베이스 구축 및 관리**: 정확하고 최신 상태의 지식 베이스를 구축하고 유지 관리하는 데 상당한 노력이 필요합니다.
*   **검색 결과의 편향성**: 검색 대상이 되는 지식 베이스 자체에 편향이 존재할 경우, RAG 시스템의 답변에도 편향이 나타날 수 있습니다.

## 5. RAG의 활용 사례

RAG 기술은 다양한 분야에서 혁신적인 솔루션을 제공하고 있습니다. 주요 활용 사례는 다음과 같습니다.

*   **질의응답 시스템**: 방대한 문서나 데이터베이스에서 관련 정보를 검색하여 사용자의 질문에 정확하고 상세하게 답변하는 데 활용됩니다. (예: 고객 지원 챗봇)
*   **콘텐츠 생성**: 특정 주제에 대한 정보를 검색한 후, 이를 바탕으로 블로그 게시물, 기사, 보고서 등 다양한 형식의 콘텐츠를 생성하는 데 사용됩니다.
*   **요약**: 긴 문서나 여러 문서를 입력받아 핵심 내용을 간결하게 요약하는 데 효과적입니다.
*   **코드 생성 및 설명**: 프로그래밍 관련 질문에 답변하거나, 특정 기능에 대한 코드를 생성하고 설명하는 데 활용될 수 있습니다.
*   **개인화 추천**: 사용자의 과거 행동이나 선호도를 기반으로 관련 정보를 검색하고, 이를 바탕으로 맞춤형 콘텐츠나 상품을 추천하는 데 적용됩니다.

이 외에도 RAG는 의료, 법률, 교육 등 전문 분야의 지식 검색 및 활용, 연구 논문 작성
==================================================